
### Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [2]:
import os 
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3.8-27b", max_tokens=800)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.1'}}, client=<groq.resources.chat.completions.Completions object at 0x1283c49e0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x128a0b920>, model_name='qwen/qwen3.8-27b', model_kwargs={}, groq_api_key=SecretStr('**********'), max_tokens=800)

In [3]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str = Field(description="The title of the movie")
    year:int = Field(description="This year the movie was released")
    director:str = Field(description="The director of the movie is")
    rating:float = Field(description="The movies rating out of 10")

In [4]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.1'}}, client=<groq.resources.chat.completions.Completions object at 0x1283c49e0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x128a0b920>, model_name='qwen/qwen3.8-27b', model_kwargs={}, groq_api_key=SecretStr('**********'), max_tokens=800), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie is', 'type': 'string'}, 'rating': {'description': 'The movies rating out of 10', 'type': 'number'}}, 'required': ['title', 'year', 'director', 'rating'], 'type': 'object'}}}], 'ls_structured_output_format': {'kwargs': {'method': 'function_calling'}, 'schema': {'type': 'function', 'function': {'name':

In [5]:
response = model_with_structure.invoke("Provide details about the moview Inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

## MEssage output alongside parsed structure

In [6]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)  

response = model_with_structure.invoke("Provide details about the movie Iron Man")
response

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '0s5epd6s8', 'function': {'arguments': '{"director":"Jon Favreau","rating":7.9,"title":"Iron Man","year":2008}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 69, 'prompt_tokens': 355, 'total_tokens': 424, 'completion_time': 0.177710215, 'completion_tokens_details': None, 'prompt_time': 0.028274301, 'prompt_tokens_details': None, 'queue_time': 0.054624997, 'total_time': 0.205984516}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_4560dae850', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0ddf9-38ee-7f03-9c1a-0dc4db28c858-0', tool_calls=[{'name': 'Movie', 'args': {'director': 'Jon Favreau', 'rating': 7.9, 'title': 'Iron Man', 'year': 2008}, 'id': '0s5epd6s8', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 355, 'output_tokens': 69, 'total_tokens': 424}

## Nested Structure 

In [7]:
class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Provide details about the movie Iron Man")
response



MovieDetails(title='Iron Man', year=2008, cast=[Actor(name='Robert Downey Jr.', role='Tony Stark / Iron Man'), Actor(name='Gwyneth Paltrow', role='Pepper Potts'), Actor(name='Jon Favreau', role='James Rhodes / War Machine'), Actor(name='Jeff Bridges', role='Obadiah Stane / Iron Monger'), Actor(name='Terrence Howard', role='Happy Hogan'), Actor(name='Sam Rockwell', role='Col. Nick Fury'), Actor(name='Faran Tahir', role='Ivan Vanko'), Actor(name='Shaun Toub', role='Ho Yinsen')], genres=['Action', 'Adventure', 'Science Fiction'], budget=140.0)

### TypedDict
TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [8]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie avengers")
response

{'director': 'Anthony Russo',
 'rating': 8.4,
 'title': 'Avengers: Endgame',
 'year': 2019}

In [9]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    # budget_: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Avengers")
response

{'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Mark Ruffalo', 'role': 'Dr. Bruce Banner / The Hulk'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'},
  {'name': 'Tom Hiddleston', 'role': 'Loki'}],
 'genres': ['Action', 'Adventure', 'Science Fiction'],
 'title': 'Avengers',
 'year': 2012}

In [11]:
model.profile